# India VAHAN Data Cleaning

Reads the raw scraped VAHAN CSVs and the supporting data (GSDP, population,
policies, charging stations) from S3, cleans them up, and merges them into
one final table. After each cleaning step, I check whether anything didn't
match up the way I expected (an unmapped state name, a vehicle class I
haven't seen before, etc.) and print it out, so I can catch problems
instead of ending up with silent NaNs.

In [ ]:
import io

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

BUCKET = "vahan-project-raw-486491621202-ap-south-1-an"
RAW_PREFIX = "scraped/"
SUPPORTING_PREFIX = "supporting_data/"
OUTPUT_PREFIX = "processed/"

s3 = boto3.client("s3")

## 1. Lookup tables

* `MONTH_MAP` — month name to month number
* `VEHICLE_CAT` — groups the raw `vehicle_class` values into broader
  categories (Car, Two Wheeler, Bus, etc.)
* `STATE_MAPPING` — cleans up all the different ways state names show up
  across the different data sources (e.g. `'Jammu & Kashmir*'`,
  `'Jammu And Kashmir'`, and `'Jammu and Kashmir'` should all become one
  thing)

In [ ]:
MONTH_MAP = {
    'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6,
    'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12,
}

VEHICLE_CAT = {
    'Agricultural Vehicle': [
        'AGRICULTURAL TRACTOR', 'POWER TILLER', 'HARVESTER',
        'TRAILER (AGRICULTURAL)', 'POWER TILLER (COMMERCIAL)',
        'TRACTOR (COMMERCIAL)', 'PULLER TRACTOR'
    ],
    'Bus': [
        'OMNI BUS (PRIVATE USE)', 'BUS', 'SCHOOL BUS',
        'EDUCATIONAL INSTITUTION BUS', 'OMNI BUS'
    ],
    'Car': ['MOTOR CAR'],
    'Construction & Industrial Equipment': [
        'FORK LIFT', 'CRANE MOUNTED VEHICLE', 'CONSTRUCTION EQUIPMENT VEHICLE',
        'ROAD ROLLER', 'EXCAVATOR (NT)', 'BULLDOZER',
        'EARTH MOVING EQUIPMENT', 'EXCAVATOR (COMMERCIAL)',
        'CONSTRUCTION EQUIPMENT VEHICLE (COMMERCIAL)'
    ],
    'Emergency Vehicle': [
        'AMBULANCE', 'ANIMAL AMBULANCE', 'FIRE TENDERS',
        'SNORKED LADDERS', 'FIRE FIGHTING VEHICLE', 'HEARSES'
    ],
    'Goods Vehicle': [
        'GOODS CARRIER', 'AUXILIARY TRAILER', 'ARTICULATED VEHICLE',
        'DUMPER', 'TRAILER (COMMERCIAL)', 'TRACTOR-TROLLEY(COMMERCIAL)',
        'SEMI-TRAILER (COMMERCIAL)', 'MODULAR HYDRAULIC TRAILER'
    ],
    'Quadricycle': ['QUADRICYCLE (PRIVATE)', 'QUADRICYCLE (COMMERCIAL)'],
    'Recreational Vehicle': [
        'CAMPER VAN / TRAILER (PRIVATE USE)', 'TRAILER FOR PERSONAL USE',
        'MOTOR CARAVAN', 'CAMPER VAN / TRAILER'
    ],
    'Service Vehicle': [
        'PRIVATE SERVICE VEHICLE (INDIVIDUAL USE)', 'PRIVATE SERVICE VEHICLE'
    ],
    'Special Purpose Vehicle': [
        'VEHICLE FITTED WITH RIG', 'VEHICLE FITTED WITH GENERATOR',
        'VEHICLE FITTED WITH COMPRESSOR', 'TOW TRUCK', 'BREAKDOWN VAN',
        'RECOVERY VEHICLE', 'TOWER WAGON', 'TREE TRIMMING VEHICLE',
        'ARMOURED/SPECIALISED VEHICLE', 'MOBILE WORKSHOP', 'CASH VAN',
        'ADAPTED VEHICLE', 'MOBILE CLINIC', 'X-RAY VAN', 'LIBRARY VAN',
        'MOBILE CANTEEN'
    ],
    'Taxi / Cab': ['LUXURY CAB', 'MAXI CAB', 'MOTOR CAB'],
    'Three Wheeler': [
        'E-RICKSHAW WITH CART (G)', 'THREE WHEELER (GOODS)',
        'THREE WHEELER (PERSONAL)', 'E-RICKSHAW(P)',
        'THREE WHEELER (PASSENGER)'
    ],
    'Two Wheeler': [
        'MOTOR CYCLE/SCOOTER-SIDECAR(T)', 'MOTOR CYCLE/SCOOTER-WITH TRAILER',
        'M-CYCLE/SCOOTER', 'M-CYCLE/SCOOTER-WITH SIDE CAR', 'MOPED',
        'MOTORISED CYCLE (CC > 25CC)', 'MOTOR CYCLE/SCOOTER-USED FOR HIRE'
    ],
    'Vintage Vehicle': ['VINTAGE MOTOR VEHICLE'],
}

# VEHICLE_CAT is grouped by category, but I need it the other way round —
# one row per vehicle_class, so I can map it onto the dataframe. Building
# it with a plain loop instead of a one-liner, just easier to read back.
VEHICLE_LOOKUP = {}
for category in VEHICLE_CAT:
    vehicle_list = VEHICLE_CAT[category]
    for vehicle in vehicle_list:
        VEHICLE_LOOKUP[vehicle] = category

print("Vehicle classes mapped:", len(VEHICLE_LOOKUP))

In [ ]:
STATE_MAPPING = {
    'All States': 'All States',
    'Andaman & Nicobar (UT)': 'Andaman and Nicobar Islands',
    'Andaman & Nicobar Island': 'Andaman and Nicobar Islands',
    'Andaman & Nicobar Islands': 'Andaman and Nicobar Islands',
    'Andaman And Nicobar Islands': 'Andaman and Nicobar Islands',
    'Andhra Pradesh': 'Andhra Pradesh',
    'Arunachal Pradesh': 'Arunachal Pradesh',
    'Assam': 'Assam',
    'Bihar': 'Bihar',
    'Chandigarh': 'Chandigarh',
    'Chandigarh (UT)': 'Chandigarh',
    'Chhattisgarh': 'Chattisgarh',
    'DNH and DD (UT)': 'Dadra & Nagar Haveli and Daman & Diu',
    'Dadra and Nagar Haveli and Daman and Diu': 'Dadra & Nagar Haveli and Daman & Diu',
    'Delhi': 'Delhi',
    'Goa': 'Goa',
    'Gujarat': 'Gujarat',
    'Haryana': 'Haryana',
    'Himachal Pradesh': 'Himachal Pradesh',
    'Jammu & Kashmir': 'Jammu and Kashmir',
    'Jammu & Kashmir*': 'Jammu and Kashmir',
    'Jammu And Kashmir': 'Jammu and Kashmir',
    'Jammu and Kashmir': 'Jammu and Kashmir',
    'Jharkhand': 'Jharkhand',
    'Karnataka': 'Karnataka',
    'Kerala': 'Kerala',
    'Ladakh': 'Ladakh',
    'Ladakh (UT)': 'Ladakh',
    'Lakshadweep': 'Lakshadweep Islands',
    'Lakshadweep (UT)': 'Lakshadweep Islands',
    'Madhya Pradesh': 'Madhya Pradesh',
    'Maharashtra': 'Maharashtra',
    'Manipur': 'Manipur',
    'Meghalaya': 'Meghalaya',
    'Mizoram': 'Mizoram',
    'NCT of Delhi': 'Delhi',
    'Nagaland': 'Nagaland',
    'Odisha': 'Odisha',
    'Puducherry': 'Pondicherry',
    'Puducherry (UT)': 'Pondicherry',
    'Punjab': 'Punjab',
    'Rajasthan': 'Rajasthan',
    'Sikkim': 'Sikkim',
    'Tamil Nadu': 'Tamil Nadu',
    'Telangana': 'Telangana',
    'Tripura': 'Tripura',
    'UT of DNH and DD': 'Dadra & Nagar Haveli and Daman & Diu',
    'Uttar Pradesh': 'Uttar Pradesh',
    'Uttarakhand': 'Uttarakhand',
    'West Bengal': 'West Bengal',
}

print("State name variants mapped:", len(STATE_MAPPING))

## 2. Load the raw VAHAN files from S3

One CSV per fuel type. I read each one, tag it with its fuel type (taken
from the filename), and stack them into one big dataframe.

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=RAW_PREFIX)

file_keys = []
for item in response['Contents']:
    if not item['Key'].endswith('/'):
        file_keys.append(item['Key'])

print("Files found:", len(file_keys))
for key in file_keys:
    print(" ", key)

In [ ]:
df_list = []

for key in file_keys:
    # filename without the folder or .csv, e.g. "scraped/petrol.csv" -> "petrol"
    fuel_name = key.split('/')[-1].replace('.csv', '')

    obj = s3.get_object(Bucket=BUCKET, Key=key)
    file_df = pd.read_csv(io.BytesIO(obj['Body'].read()))

    print(fuel_name, "-", len(file_df), "rows -", list(file_df.columns))

    file_df['fuel_type'] = fuel_name
    df_list.append(file_df)

combined_df = pd.concat(df_list, ignore_index=True)
print()
print("Combined shape:", combined_df.shape)

Quick check — the combined row count should equal the sum of what was
printed for each file above. If it doesn't, something went wrong in the
concat (e.g. mismatched columns between files).

In [ ]:
rows_added_up = sum(len(d) for d in df_list)
print("Sum of individual files:", rows_added_up)
print("Combined dataframe rows:", len(combined_df))

if rows_added_up == len(combined_df):
    print("Matches, good.")
else:
    print("These don't match — need to check the columns in each file.")

## 3. A quick look at the raw data before cleaning

Just `.isna().sum()` and `.duplicated().sum()` — nothing fancy, just
wanted a baseline to compare against once cleaning is done.

In [ ]:
print(combined_df.isna().sum())
print()
print("Exact duplicate rows:", combined_df.duplicated().sum())

In [ ]:
plt.figure(figsize=(8, 4))
sns.heatmap(combined_df.isna(), cbar=False, yticklabels=False, cmap="rocket_r")
plt.title("Missing values in raw data (dark = missing)")
plt.tight_layout()
plt.show()

## 4. Clean up the main columns

Keep only the columns I need, and clean up `value` (it comes in as text
with commas, e.g. `"1,234"`).

In [ ]:
df = combined_df.copy()
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df[['year', 'month_wise', 'state', 'vehicle_class', 'fuel_type', 'value']]

df['value'] = df['value'].astype(str).str.replace(',', '')
df['value'] = pd.to_numeric(df['value'], errors='coerce')

df.head()

`errors='coerce'` turns anything that isn't a number into `NaN` — so I
want to check whether that actually happened to any real values (as
opposed to `value` already being blank).

In [ ]:
bad_values = df[df['value'].isna() & combined_df['value'].notna()]
print("Rows where 'value' failed to convert to a number:", len(bad_values))

if len(bad_values) > 0:
    print(bad_values['value'].head(10))
    print(combined_df.loc[bad_values.index, 'value'].value_counts().head(10))

### Map vehicle class to category

Check whether every `vehicle_class` in the data is actually covered by
`VEHICLE_LOOKUP` — if VAHAN adds a new vehicle type I haven't seen before,
this is where it would show up as unmapped (and become NaN otherwise).

In [ ]:
df['vehicle_category'] = df['vehicle_class'].map(VEHICLE_LOOKUP)

unmapped = df[df['vehicle_category'].isna() & df['vehicle_class'].notna()]
print("Rows with an unmapped vehicle_class:", len(unmapped))

if len(unmapped) > 0:
    print(unmapped['vehicle_class'].value_counts())
else:
    print("Every vehicle_class matched something in VEHICLE_LOOKUP.")

### Map month name to month number

In [ ]:
df['month_number'] = df['month_wise'].map(MONTH_MAP)

unmapped_months = df[df['month_number'].isna() & df['month_wise'].notna()]
print("Rows with an unmapped month:", len(unmapped_months))

if len(unmapped_months) > 0:
    print(unmapped_months['month_wise'].unique())

### Work out the financial year

India's financial year runs April to March. So month 4 (April) onward
belongs to the year that's starting; months 1–3 (Jan–Mar) belong to the
year that's ending.

In [ ]:
df['financial_year'] = np.where(
    df['month_number'] >= 4,
    df['year'].astype(str) + '-' + (df['year'] + 1).astype(str).str[2:],
    (df['year'] - 1).astype(str) + '-' + df['year'].astype(str).str[2:],
)

print("Any missing financial_year values:", df['financial_year'].isna().sum())
df[['year', 'month_wise', 'month_number', 'financial_year']].drop_duplicates().sort_values(['year', 'month_number']).head(10)

### Clean up state names

This is the trickiest one — the same state shows up spelled differently
across the different files (`'Jammu & Kashmir*'`, `'Jammu And Kashmir'`,
etc.), so I built `STATE_MAPPING` by hand after checking what raw values
actually show up. If a new spelling shows up that isn't in the dictionary
yet, it'll turn into NaN — so checking for that here first.

In [ ]:
raw_states = df['state'].unique()

not_in_mapping = []
for s in raw_states:
    if s not in STATE_MAPPING:
        not_in_mapping.append(s)

print("Raw state values not covered by STATE_MAPPING:", not_in_mapping)

In [ ]:
df['state'] = df['state'].map(STATE_MAPPING)

print("Rows with missing state after mapping:", df['state'].isna().sum())

## 5. Supporting data — GSDP, population, policies, charging stations

Same idea — load each one, clean up the state names the same way.

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key=SUPPORTING_PREFIX + "gsdp.csv")
gsdp = pd.read_csv(io.BytesIO(obj['Body'].read()))

gsdp_melted = gsdp.melt(id_vars="State/Union Territory", var_name="financial_year", value_name="gsdp_lakhs")
gsdp_melted['gsdp_lakhs'] = gsdp_melted['gsdp_lakhs'].astype(str).str.replace(',', '')
gsdp_melted['gsdp_lakhs'] = pd.to_numeric(gsdp_melted['gsdp_lakhs'], errors='coerce')

gsdp_melted = gsdp_melted.rename(columns={'State/Union Territory': 'state'})

print("GSDP state values not covered by STATE_MAPPING:")
print([s for s in gsdp_melted['state'].unique() if s not in STATE_MAPPING])

gsdp_melted['state'] = gsdp_melted['state'].map(STATE_MAPPING)
gsdp_melted.head()

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key=SUPPORTING_PREFIX + "support.xlsx")
excel_bytes = io.BytesIO(obj['Body'].read())

all_sheets = pd.read_excel(excel_bytes, sheet_name=None)
print("Sheets in the workbook:", list(all_sheets.keys()))

population = all_sheets['population']
policies = all_sheets['policies']
charging_stations = all_sheets['charging_stations']
state_codes = all_sheets['state_codes']

In [ ]:
population = population.rename(columns={'State/Union Territory': 'State'})

print("Population state values not covered by STATE_MAPPING:")
print([s for s in population['State'].unique() if s not in STATE_MAPPING])

population['State'] = population['State'].map(STATE_MAPPING)
population_melted = population.melt(id_vars="State", var_name="year", value_name="population")
population_melted['year'] = population_melted['year'].astype(int)
population_melted.head()

In [ ]:
print("Policies — state values not covered by STATE_MAPPING:")
print([s for s in policies['State'].unique() if s not in STATE_MAPPING])
policies['State'] = policies['State'].map(STATE_MAPPING)

print()
print("Charging stations — state values not covered by STATE_MAPPING:")
print([s for s in charging_stations['State'].unique() if s not in STATE_MAPPING])
charging_stations['State'] = charging_stations['State'].map(STATE_MAPPING)

charging_stations.head()

Note: `policies` and `charging_stations` are cleaned up here (state
names normalized) but I'm **not** merging them into `final_df` below —
they don't share a clean join key with the registration data (policies is
one row per state, not per year/month; charging stations is a single
snapshot in time). They get exported separately at the end instead, for
their own tables in the database.

## 6. Merge everything together

Left join `state_codes`, `population`, then `gsdp` onto the cleaned VAHAN
data. Checking the row count after each merge — a left join should never
add or remove rows from the left-hand table, so if the count changes,
something's duplicated on the right-hand side.

In [ ]:
rows_before = len(df)
print("Rows before merging:", rows_before)

final_df = df.merge(state_codes, how='left', left_on='state', right_on='State')
final_df = final_df.drop(columns='State')
print("Rows after merging state_codes:", len(final_df))

In [ ]:
final_df = final_df.merge(population_melted, how='left', left_on=['state', 'year'], right_on=['State', 'year'])
final_df = final_df.drop(columns='State')
print("Rows after merging population:", len(final_df))

In [ ]:
final_df = final_df.merge(gsdp_melted, how='left', on=['state', 'financial_year'])
print("Rows after merging gsdp:", len(final_df))

if len(final_df) == rows_before:
    print("Row count is unchanged from before merging — good.")
else:
    print("Row count changed! One of the merges duplicated rows somewhere.")

In [ ]:
final_df.columns = (
    final_df.columns.str.replace('/', '_')
    .str.replace(' ', '_')
    .str.lower()
    .str.replace('tin', 'state_id')
    .str.replace('month_wise', 'month')
    .str.replace('value', 'registrations')
)

final_df = final_df[[
    'year', 'month_number', 'month', 'financial_year', 'state_id', 'state_code',
    'state', 'population', 'gsdp_lakhs', 'fuel_type', 'vehicle_class',
    'vehicle_category', 'registrations',
]]

final_df.head()

## 7. Final checks before saving

A few last sanity checks: no duplicate rows at the (state, year, month,
fuel_type, vehicle_class) level, no negative registration counts, and a
look at where the remaining nulls are.

In [ ]:
dupe_count = final_df.duplicated(subset=['state', 'year', 'month_number', 'fuel_type', 'vehicle_class']).sum()
print("Duplicate rows at the (state, year, month, fuel_type, vehicle_class) level:", dupe_count)

negative_count = (final_df['registrations'] < 0).sum()
print("Rows with negative registrations:", negative_count)

In [ ]:
null_counts = final_df.isna().sum()
null_pct = (null_counts / len(final_df) * 100).round(2)

pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})

`gsdp_lakhs` being null for the most recent financial years is
expected — GSDP figures get released with a lag, so the newest periods
genuinely don't have a number yet. Nulls in `state_id`, `state_code`, or
`population` would be more concerning and worth tracing back through
section 5/6 above.

In [ ]:
plt.figure(figsize=(8, 4))
final_df['registrations'].clip(lower=1).plot.hist(bins=50, logx=True)
plt.title("Distribution of registrations (log scale)")
plt.xlabel("Registrations")
plt.tight_layout()
plt.show()

## 8. Save the cleaned data back to S3

In [ ]:
def save_to_s3(dataframe, key):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, index=False)
    s3.put_object(Bucket=BUCKET, Key=key, Body=csv_buffer.getvalue())
    print("Saved", key, "-", len(dataframe), "rows")

save_to_s3(final_df, OUTPUT_PREFIX + "final_df.csv")
save_to_s3(policies, OUTPUT_PREFIX + "policies.csv")
save_to_s3(charging_stations, OUTPUT_PREFIX + "charging_stations.csv")

## Notes

Things I actually found when I ran this:

- Any unmapped states, vehicle classes, or months? (sections 4–5)
- How much of `gsdp_lakhs` ended up null, and for which years?
- Any duplicate rows in the final check (section 7)?
